# Tagging and Extraction Using OpenAI Functions

In [1]:
from dotenv import load_dotenv, find_dotenv
import os
import openai

_ = load_dotenv(find_dotenv())
openai.api_key = os.environ["OPENAI_API_KEY"]


In [2]:
from langchain.chat_models import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain.utils.openai_functions import convert_pydantic_to_openai_function
from pydantic import BaseModel, Field


## Tagging

In [3]:
class Tagging(BaseModel):
    """Tag the piece of text with particular information"""
    sentiment: str = Field(description = "sentiment of text, should be 'pos', 'neg' or 'neutral' for positive, negative and neutral text respectively")
    lang: str = Field(description = "language of the text(should be ISO 639-1 code)")


In [4]:
tagging_function = convert_pydantic_to_openai_function(Tagging)


In [5]:
model = ChatOpenAI()


In [6]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "Think carefully and tag the text as instructed."),
    ("user", "{input}")
])


Since I don't want the LLM to get confused and just reply to the text instead of tagging it, I'll force the model to call the function on the input.

In [7]:
model_with_func = model.bind(functions = [tagging_function], function_call = {"name": "Tagging"})


In [8]:
chain = prompt | model_with_func


In [9]:
chain.invoke({"input": "Working with AI is fun"})


AIMessage(content='', additional_kwargs={'function_call': {'name': 'Tagging', 'arguments': '{"sentiment":"pos","lang":"en"}'}})

In [10]:
from langchain.output_parsers.openai_functions import JsonOutputFunctionsParser


In [11]:
new_chain = prompt | model_with_func | JsonOutputFunctionsParser()


In [12]:
new_chain.invoke({"input": "non mi piace questo cibo"})


{'sentiment': 'neg', 'lang': 'it'}

## Extraction

Extraction is similar to tagging, but used for extracting multiple pieces of information.

In [13]:
from typing import Optional


In [14]:
class Person(BaseModel):
    """Information about a person"""
    Name : str = Field(description = "name of the person")
    age : Optional[int] = Field(description = "age of the person")


In [15]:
class Information(BaseModel):
    """Information to extract"""
    people: list[Person] = Field(description = "information about people")


In [16]:
convert_pydantic_to_openai_function(Information)


{'name': 'Information',
 'description': 'Information to extract',
 'parameters': {'title': 'Information',
  'description': 'Information to extract',
  'type': 'object',
  'properties': {'people': {'title': 'People',
    'description': 'information about people',
    'type': 'array',
    'items': {'title': 'Person',
     'description': 'Information about a person',
     'type': 'object',
     'properties': {'Name': {'title': 'Name',
       'description': 'name of the person',
       'type': 'string'},
      'age': {'title': 'Age',
       'description': 'age of the person',
       'type': 'integer'}},
     'required': ['Name']}}},
  'required': ['people']}}

In [17]:
extract_func = convert_pydantic_to_openai_function(Information)
extract_model = model.bind(functions = [extract_func], function_call = {"name": "Information"})


In [18]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "Extract the relevant information, if not provided then don't assume any arbitrary values, just extract partial info."),
    ("user", "{input}")
])


In [19]:
chain = prompt | extract_model | JsonOutputFunctionsParser()


In [20]:
chain.invoke({"input": "Joe is 30 years old and his mother's name is Martha"})


{'people': [{'Name': 'Joe', 'age': 30}, {'Name': 'Martha'}]}

You can use `JsonKeyOutputFunctionsParser` if you only care about the extracted values and don't want the top-level keyword wrapping them.

In [21]:
from langchain.output_parsers.openai_functions import JsonKeyOutputFunctionsParser


In [22]:
new_chain = prompt | extract_model | JsonKeyOutputFunctionsParser(key_name = "people")


In [23]:
new_chain.invoke({"input": "Joe is 30 years old and his mother's name is Martha"})


[{'Name': 'Joe', 'age': 30}, {'Name': 'Martha'}]

## Real life examples of tagging and extraction

In [24]:
from langchain.document_loaders import WebBaseLoader

In [25]:
loader = WebBaseLoader("https://lilianweng.github.io/posts/2023-06-23-agent/")
documents = loader.load()
documents[0].page_content[:1000]

"\n\n\n\n\n\nLLM Powered Autonomous Agents | Lil'Log\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nLil'Log\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n|\n\n\n\n\n\n\nPosts\n\n\n\n\nArchive\n\n\n\n\nSearch\n\n\n\n\nTags\n\n\n\n\nFAQ\n\n\n\n\n\n\n\n\n\n      LLM Powered Autonomous Agents\n    \nDate: June 23, 2023  |  Estimated Reading Time: 31 min  |  Author: Lilian Weng\n\n\n \n\n\nTable of Contents\n\n\n\nAgent System Overview\n\nComponent One: Planning\n\nTask Decomposition\n\nSelf-Reflection\n\n\nComponent Two: Memory\n\nTypes of Memory\n\nMaximum Inner Product Search (MIPS)\n\n\nComponent Three: Tool Use\n\nCase Studies\n\nScientific Discovery Agent\n\nGenerative Agents Simulation\n\nProof-of-Concept Examples\n\n\nChallenges\n\nCitation\n\nReferences\n\n\n\n\n\nBuilding agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The pot

In [26]:
page_content = documents[0].page_content[:1000] #we will give the above text to the model to extract some information

In [27]:
class overview(BaseModel):
    """overview of a section of the text"""
    summary: str = Field(description = "provide concise summary of the text")
    language: str = Field(description = "provide the language of the text that it is written in")
    keywords: str = Field(description = "write down the keywords related to the text")

In [28]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "extract relevant information as instructed"),
    ("user", "{input}")
])

In [29]:
tagging_func = [convert_pydantic_to_openai_function(overview)]
model = ChatOpenAI(temperature = 0)
model_with_func = model.bind(functions = tagging_func, function_call = {"name": "overview"})
chain = prompt | model_with_func | JsonOutputFunctionsParser()

In [30]:
chain.invoke({"input": page_content})

{'summary': 'The text discusses the concept of building autonomous agents powered by LLM (large language model) as the core controller. It explores components like planning, memory, and tool use, along with proof-of-concept examples and challenges. The potential of LLM goes beyond generating content to being a general problem solver.',
 'language': 'English',
 'keywords': 'LLM, autonomous agents, planning, memory, tool use, proof-of-concept, challenges, general problem solver'}

Now we'll try to get the paper names and author names from the LLM, given the text.

In [31]:
class Paper(BaseModel):
    """Information about papers mentioned."""
    title: str = Field(description = "provide title of the paper")
    author: Optional[str] = Field(description = "provide name of the author")


class Info(BaseModel):
    """Information to extract"""
    papers: list[Paper] = Field(description = "information about the papers")

In [32]:
system_prompt = """A article will be passed to you. Extract from it all papers that are mentioned by this article follow by its author. 

Do not extract the name of the article itself. If no papers are mentioned that's fine - you don't need to extract any! Just return an empty list.

Do not make up or guess ANY extra information. Only extract what exactly is in the text."""

new_prompt = ChatPromptTemplate.from_messages(
[
    ("system", system_prompt),
    ("user", "{input}")
])

In [33]:
info_func = [convert_pydantic_to_openai_function(Info)]
model_with_info = model.bind(functions = info_func, function_call = {"name": "Info"})
chain = new_prompt | model_with_info | JsonOutputFunctionsParser()

In [34]:
page_content = documents[0].page_content[:10000]

In [35]:
info_func

[{'name': 'Info',
  'description': 'Information to extract',
  'parameters': {'title': 'Info',
   'description': 'Information to extract',
   'type': 'object',
   'properties': {'papers': {'title': 'Papers',
     'description': 'information about the papers',
     'type': 'array',
     'items': {'title': 'Paper',
      'description': 'Information about papers mentioned.',
      'type': 'object',
      'properties': {'title': {'title': 'Title',
        'description': 'provide title of the paper',
        'type': 'string'},
       'author': {'title': 'Author',
        'description': 'provide name of the author',
        'type': 'string'}},
      'required': ['title']}}},
   'required': ['papers']}}]

In [36]:
chain.invoke({"input": page_content})

{'papers': [{'title': 'Chain of thought (CoT; Wei et al. 2022)',
   'author': 'Wei et al. 2022'},
  {'title': 'Tree of Thoughts (Yao et al. 2023)', 'author': 'Yao et al. 2023'},
  {'title': 'LLM+P (Liu et al. 2023)', 'author': 'Liu et al. 2023'},
  {'title': 'ReAct (Yao et al. 2023)', 'author': 'Yao et al. 2023'},
  {'title': 'Reflexion (Shinn & Labash 2023)',
   'author': 'Shinn & Labash 2023'},
  {'title': 'Chain of Hindsight (CoH; Liu et al. 2023)',
   'author': 'Liu et al. 2023'},
  {'title': 'Algorithm Distillation (AD; Laskin et al. 2023)',
   'author': 'Laskin et al. 2023'}]}

## Splitting the whole document with `RecursiveCharacterTextSplitter`

What if instead of just the first 10,000 characters, we want the LLM to read the whole document? A single prompt still can't fit the entire thing, so instead we split the document into smaller chunks and process each one.

`RecursiveCharacterTextSplitter` does this splitting for us:
* You give it a `chunk_size` (max characters per chunk) and a `chunk_overlap` (how many characters get repeated between consecutive chunks)
* It tries splitting on a prioritized list of separators - by default something like paragraphs (`\n\n`), then lines (`\n`), then words, then characters - always trying the "biggest" natural break first, and only falling back to a smaller one if a chunk is still too big
* This way it keeps each chunk as semantically coherent as possible (e.g. it'd rather split between paragraphs than in the middle of a sentence), instead of blindly cutting the text every N characters
* The result is a list of text chunks, each small enough to send to the LLM individually - which is exactly what we do next with `.map()`


In [37]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [38]:
text_splitter = RecursiveCharacterTextSplitter(chunk_overlap = 0) #chunk_overlap controls how many characters from the end of one chunk get repeated at the start of the next - useful for preserving context that straddles a chunk boundary. It's set to 0 here since we don't want any repeated content, so a paper mentioned near a chunk boundary doesn't end up getting extracted twice


In [39]:
doc = documents[0]

In [40]:
splits = text_splitter.split_text(doc.page_content)

In [41]:
len(splits)

15

In [42]:
type(splits)

list

In [43]:
len(splits[0])

2154

### How the split happened

We didn't pass a `chunk_size`, so `RecursiveCharacterTextSplitter` used its default (4000 characters). The whole document got broken into 15 chunks (`len(splits) == 15`), each one up to that size - the first chunk (`splits[0]`) came out to 2154 characters, since the splitter looks for the nearest natural break point (paragraph, then line, then word) at or before the 4000-character limit, rather than cutting off exactly at 4000 mid-sentence.


In [44]:
def flatten(matrix):
    flat_list = []
    for rows in matrix:
        flat_list += rows
    return flat_list

In [45]:
flatten([[1,2], [3, 4]])

[1, 2, 3, 4]

In [46]:
from langchain.schema.runnable import RunnableLambda

In [47]:
prep = RunnableLambda(
    lambda x: [{"input": doc} for doc in text_splitter.split_text(x)]
)

In [51]:
extraction_chain = new_prompt | model_with_info | JsonKeyOutputFunctionsParser(key_name = "papers")

In [52]:
chain = prep| extraction_chain.map()| flatten

In [53]:
chain.invoke(doc.page_content)

[{'title': 'AutoGPT', 'author': 'Unknown'},
 {'title': 'GPT-Engineer', 'author': 'Unknown'},
 {'title': 'BabyAGI', 'author': 'Unknown'},
 {'title': 'Chain of thought', 'author': 'Wei et al. 2022'},
 {'title': 'Tree of Thoughts', 'author': 'Yao et al. 2023'},
 {'title': 'LLM+P', 'author': 'Liu et al. 2023'},
 {'title': 'ReAct', 'author': 'Yao et al. 2023'},
 {'title': 'Reflexion', 'author': 'Shinn & Labash 2023'},
 {'title': 'Chain of Hindsight (CoH)', 'author': 'Liu et al. 2023'},
 {'title': 'Algorithm Distillation (AD)', 'author': 'Laskin et al. 2023'},
 {'title': 'Duan et al. 2017'},
 {'title': 'Laskin et al. 2023'},
 {'title': 'Miller 1956'},
 {'title': 'LSH (Locality-Sensitive Hashing)',
  'author': 'Andoni, Alexandr and Indyk, Piotr'},
 {'title': 'ANNOY (Approximate Nearest Neighbors Oh Yeah)'},
 {'title': 'HNSW (Hierarchical Navigable Small World)'},
 {'title': 'FAISS (Facebook AI Similarity Search)'},
 {'title': 'ScaNN (Scalable Nearest Neighbors)'},
 {'title': 'MRKL', 'author':